# Credit Card Fraud Detection — Base Models

Trains all four base models and saves their predictions to disk for use by `fraud_detection.ipynb`.

**Output files:**
- `ffn_train_oof.npy`, `ffn_val_probs.npy`, `ffn_test_probs.npy`
- `xgb_train_oof.npy`, `xgb_val_probs.npy`, `xgb_test_probs.npy`
- `ae_train_probs.npy`, `ae_val_probs.npy`, `ae_test_probs.npy`
- `if_train_probs.npy`, `if_val_probs.npy`, `if_test_probs.npy`
- `y_train.npy`, `y_val.npy`, `y_test.npy`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv("creditcard.csv")

print(f"Shape: {df.shape}")
counts = df["Class"].value_counts()
print(f"\nClass distribution:")
print(f"  Normal (0): {counts[0]:,}  ({counts[0]/len(df)*100:.2f}%)")
print(f"  Fraud  (1): {counts[1]:,}  ({counts[1]/len(df)*100:.4f}%)")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Normal", "Fraud"], [counts[0], counts[1]], color=["steelblue", "crimson"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center")

df[df["Class"] == 0]["Amount"].hist(ax=axes[1], bins=50, alpha=0.6, label="Normal", color="steelblue")
df[df["Class"] == 1]["Amount"].hist(ax=axes[1], bins=50, alpha=0.6, label="Fraud",  color="crimson")
axes[1].set_title("Transaction Amount by Class")
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
scaler = StandardScaler()
df["scaled_Amount"] = scaler.fit_transform(df[["Amount"]])
df["scaled_Time"]   = scaler.fit_transform(df[["Time"]])

feature_cols = [c for c in df.columns if c.startswith("V")] + ["scaled_Amount", "scaled_Time"]
X = df[feature_cols].values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

INPUT_DIM = X_train.shape[1]

print(f"Train: {X_train.shape[0]:,}  ({int(y_train.sum())} fraud)")
print(f"Val:   {X_val.shape[0]:,}   ({int(y_val.sum())} fraud)")
print(f"Test:  {X_test.shape[0]:,}   ({int(y_test.sum())} fraud)")
print(f"Input dim: {INPUT_DIM}")

## 3. Base Models

### Model Definitions

In [ ]:
class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class FFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.BatchNorm1d(64),  nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def get_recon_errors(model, X, batch_size=512):
    model.eval()
    errors = []
    X_t = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb = X_t[i:i + batch_size].to(DEVICE)
            mse = ((xb - model(xb)) ** 2).mean(dim=1).cpu().numpy()
            errors.append(mse)
    return np.concatenate(errors)


print("Classes defined.")

### 3.1 FFN (Supervised)

In [ ]:
train_ds = FraudDataset(X_train, y_train)
val_ds   = FraudDataset(X_val,   y_val)
test_ds  = FraudDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(DEVICE)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── K-fold OOF ────────────────────────────────────────────────────────────
ffn_train_oof = np.zeros(len(X_train), dtype=np.float32)
print(f"FFN — {N_FOLDS}-fold OOF (10 epochs/fold):")
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
    y_tr = y_train[tr_idx]
    fold_ds     = FraudDataset(X_tr, y_tr)
    fold_loader = DataLoader(fold_ds, batch_size=256, shuffle=True, num_workers=0)
    neg_f, pos_f = (y_tr == 0).sum(), (y_tr == 1).sum()
    fold_pw   = torch.tensor([neg_f / pos_f], dtype=torch.float32).to(DEVICE)
    fold_model = FFN(INPUT_DIM).to(DEVICE)
    fold_opt   = torch.optim.Adam(fold_model.parameters(), lr=1e-3, weight_decay=1e-5)
    fold_crit  = nn.BCEWithLogitsLoss(pos_weight=fold_pw)
    for _ in range(10):
        fold_model.train()
        for xb, yb in fold_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            fold_opt.zero_grad(); fold_crit(fold_model(xb), yb).backward(); fold_opt.step()
    fold_model.eval()
    with torch.no_grad():
        xvl_t = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)
        ffn_train_oof[vl_idx] = torch.sigmoid(fold_model(xvl_t)).cpu().numpy()
    print(f"  Fold {fold+1}/{N_FOLDS}  OOF fraud mean={ffn_train_oof[vl_idx][y_train[vl_idx]==1].mean():.3f}")

# ── Full retrain ──────────────────────────────────────────────────────────
print("\nFFN — full retrain (20 epochs):")
ffn = FFN(INPUT_DIM).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(ffn.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
best_val_loss = float("inf")
ffn_train_losses, ffn_val_losses = [], []

for epoch in range(1, 21):
    ffn.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(); loss = criterion(ffn(xb), yb); loss.backward(); optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_ds)
    ffn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_loss += criterion(ffn(xb), yb).item() * len(xb)
    val_loss /= len(val_ds)
    scheduler.step(val_loss)
    ffn_train_losses.append(train_loss)
    ffn_val_losses.append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss; torch.save(ffn.state_dict(), "ffn_best.pt")
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/20  train={train_loss:.4f}  val={val_loss:.4f}")
print("Done.")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ffn_train_losses, label="Train Loss")
plt.plot(ffn_val_losses,   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("BCE Loss"); plt.title("FFN Training Curve")
plt.legend(); plt.tight_layout(); plt.show()

ffn.load_state_dict(torch.load("ffn_best.pt", map_location=DEVICE))

def get_ffn_probs(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            probs.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
    return np.concatenate(probs)

ffn_val_probs  = get_ffn_probs(ffn, val_loader)
ffn_test_probs = get_ffn_probs(ffn, test_loader)
print(f"FFN val  fraud mean={ffn_val_probs[y_val==1].mean():.4f}")
print(f"FFN test fraud mean={ffn_test_probs[y_test==1].mean():.4f}")

### 3.2 XGBoost (Supervised)

In [ ]:
xgb_params = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                  scale_pos_weight=neg/pos, eval_metric="aucpr",
                  random_state=SEED, n_jobs=-1, verbosity=0)

xgb_train_oof = np.zeros(len(X_train), dtype=np.float32)
print(f"XGBoost — {N_FOLDS}-fold OOF:")
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_train, y_train)):
    clf_fold = xgb.XGBClassifier(**xgb_params)
    clf_fold.fit(X_train[tr_idx], y_train[tr_idx])
    xgb_train_oof[vl_idx] = clf_fold.predict_proba(X_train[vl_idx])[:, 1]
    print(f"  Fold {fold+1}/{N_FOLDS}  OOF fraud mean={xgb_train_oof[vl_idx][y_train[vl_idx]==1].mean():.3f}")

xgb_full = xgb.XGBClassifier(**xgb_params)
xgb_full.fit(X_train, y_train)

xgb_val_probs  = xgb_full.predict_proba(X_val)[:, 1]
xgb_test_probs = xgb_full.predict_proba(X_test)[:, 1]
print(f"\nXGBoost val  fraud mean={xgb_val_probs[y_val==1].mean():.4f}")
print(f"XGBoost test fraud mean={xgb_test_probs[y_test==1].mean():.4f}")

### 3.3 Autoencoder (Unsupervised)

In [ ]:
X_train_normal = X_train[y_train == 0]
normal_ds      = FraudDataset(X_train_normal, np.zeros(len(X_train_normal), dtype=np.float32))
normal_loader  = DataLoader(normal_ds, batch_size=256, shuffle=True, num_workers=0)

ae = Autoencoder(INPUT_DIM).to(DEVICE)
ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
ae_criterion = nn.MSELoss()
ae_losses = []

print("Autoencoder — training (20 epochs):")
for epoch in range(1, 21):
    ae.train()
    epoch_loss = 0.0
    for xb, _ in normal_loader:
        xb = xb.to(DEVICE)
        ae_optimizer.zero_grad(); loss = ae_criterion(ae(xb), xb); loss.backward(); ae_optimizer.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(normal_ds)
    ae_losses.append(epoch_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/20  recon_loss={epoch_loss:.6f}")

torch.save(ae.state_dict(), "ae_best.pt")

ae_err_scaler = StandardScaler()
ae_train_probs = sigmoid(ae_err_scaler.fit_transform(get_recon_errors(ae, X_train).reshape(-1,1)).ravel())
ae_val_probs   = sigmoid(ae_err_scaler.transform(get_recon_errors(ae, X_val).reshape(-1,1)).ravel())
ae_test_probs  = sigmoid(ae_err_scaler.transform(get_recon_errors(ae, X_test).reshape(-1,1)).ravel())

print(f"\nAE train fraud mean={ae_train_probs[y_train==1].mean():.4f}  normal={ae_train_probs[y_train==0].mean():.4f}")
print(f"AE val   fraud mean={ae_val_probs[y_val==1].mean():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ae_val_errors = get_recon_errors(ae, X_val)
ax.hist(ae_val_errors[y_val==0], bins=80, alpha=0.6, label="Normal", color="steelblue", density=True)
ax.hist(ae_val_errors[y_val==1], bins=80, alpha=0.6, label="Fraud",  color="crimson",   density=True)
ax.set_xlabel("Reconstruction Error (MSE)"); ax.set_ylabel("Density")
ax.set_title("Autoencoder Reconstruction Error — Validation Set")
ax.legend(); ax.set_xlim(0, np.percentile(ae_val_errors, 99.5))
plt.tight_layout(); plt.show()

### 3.4 Isolation Forest (Unsupervised)

In [ ]:
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1)
iso.fit(X_train_normal)

if_scaler = StandardScaler()
if_train_probs = sigmoid(if_scaler.fit_transform((-iso.score_samples(X_train)).reshape(-1,1)).ravel())
if_val_probs   = sigmoid(if_scaler.transform((-iso.score_samples(X_val)).reshape(-1,1)).ravel())
if_test_probs  = sigmoid(if_scaler.transform((-iso.score_samples(X_test)).reshape(-1,1)).ravel())

print(f"IF train fraud mean={if_train_probs[y_train==1].mean():.4f}  normal={if_train_probs[y_train==0].mean():.4f}")
print(f"IF val   fraud mean={if_val_probs[y_val==1].mean():.4f}")

if_val_scores = -iso.score_samples(X_val)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(if_val_scores[y_val==0], bins=80, alpha=0.6, label="Normal", color="steelblue", density=True)
ax.hist(if_val_scores[y_val==1], bins=80, alpha=0.6, label="Fraud",  color="crimson",   density=True)
ax.set_xlabel("Anomaly Score"); ax.set_ylabel("Density")
ax.set_title("Isolation Forest Anomaly Scores — Validation Set")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Save Outputs to Disk

In [ ]:
# FFN
np.save("ffn_train_oof.npy",  ffn_train_oof)
np.save("ffn_val_probs.npy",  ffn_val_probs)
np.save("ffn_test_probs.npy", ffn_test_probs)

# XGBoost
np.save("xgb_train_oof.npy",  xgb_train_oof)
np.save("xgb_val_probs.npy",  xgb_val_probs)
np.save("xgb_test_probs.npy", xgb_test_probs)

# Autoencoder
np.save("ae_train_probs.npy", ae_train_probs)
np.save("ae_val_probs.npy",   ae_val_probs)
np.save("ae_test_probs.npy",  ae_test_probs)

# Isolation Forest
np.save("if_train_probs.npy", if_train_probs)
np.save("if_val_probs.npy",   if_val_probs)
np.save("if_test_probs.npy",  if_test_probs)

# Labels
np.save("y_train.npy", y_train)
np.save("y_val.npy",   y_val)
np.save("y_test.npy",  y_test)

print("All outputs saved. Run fraud_detection.ipynb next.")